# Generador de manifiestos Kubernetes

Este notebook genera YAML basico para `Deployment` y `Service` sin depender de librerias externas. Es util para entender la estructura del manifiesto antes de introducir herramientas mas avanzadas.


In [ ]:
from textwrap import dedent


def render_env(env):
    if not env:
        return ""
    lines = ["          env:"]
    for key, value in env.items():
        lines.append(f"            - name: {key}")
        lines.append(f"              value: \"{value}\"")
    return "\n".join(lines)


def render_deployment(name, image, container_port, replicas=2, env=None):
    env_block = render_env(env)
    if env_block:
        env_block = env_block + "\n"
    return dedent(
        f"""
        apiVersion: apps/v1
        kind: Deployment
        metadata:
          name: {name}
        spec:
          replicas: {replicas}
          selector:
            matchLabels:
              app: {name}
          template:
            metadata:
              labels:
                app: {name}
            spec:
              containers:
                - name: {name}
                  image: {image}
{env_block}                  ports:
                    - containerPort: {container_port}
        """
    ).strip()


def render_service(name, service_port, target_port):
    return dedent(
        f"""
        apiVersion: v1
        kind: Service
        metadata:
          name: {name}
        spec:
          selector:
            app: {name}
          ports:
            - port: {service_port}
              targetPort: {target_port}
          type: ClusterIP
        """
    ).strip()


In [ ]:
deployment_yaml = render_deployment(
    name="mi-api",
    image="usuario/mi-api:0.1.0",
    container_port=8000,
    replicas=3,
    env={"APP_ENV": "production", "LOG_LEVEL": "info"},
)

service_yaml = render_service(
    name="mi-api",
    service_port=80,
    target_port=8000,
)

print(deployment_yaml)
print()
print(service_yaml)


## Ideas para extenderlo

- Generar `ConfigMap` y `Secret` a partir de diccionarios.
- Anadir `resources`, `readinessProbe` y `livenessProbe`.
- Escribir el YAML a disco para llevarlo a `examples/`.
